# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR<sup>2</sup> dataset using the `mlcroissant` library, following the Croissant schema standard.

### Dataset Source
The dataset source is provided via a Croissant schema at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

For reference, **all Croissant entities such as record sets and fields are referenced by their `@id` fields**.

In [ ]:
# Ensure `mlcroissant` is installed in the current environment
!pip install -U mlcroissant

## 1. Data Loading

We load the Croissant schema and dataset metadata using `mlcroissant`. This provides access to all schema components: record sets, fields, and file objects.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset schema and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Dataset Description: {metadata.description}")

## 2. Data Overview

Let's inspect the list of available record sets (`cr:RecordSet`), their `@id`, and the fields described in the Croissant schema.

In [ ]:
# List all RecordSets in the dataset metadata schema.
record_sets = metadata.record_sets
print(f"Record sets found ({len(record_sets)}):")
for record_set in record_sets:
    print(f"- @id: {record_set.id} | Name: {record_set.name}")

# Show fields of each RecordSet
for record_set in record_sets:
    print(f"\nFields for RecordSet '@id': {record_set.id} | {record_set.name}")
    for field in record_set.fields:
        print(f"  - field @id: {field.id} | Name: {field.name} | Description: {field.description}")

## 3. Data Extraction

Extract data from a specific record set into a DataFrame using the provided `@id` references.

Below, we extract all record sets into separate DataFrames using their `@id`.

In [ ]:
# Gather all RecordSet @id values
record_set_ids = [rs.id for rs in metadata.record_sets]
print("RecordSet @id values:")
print(record_set_ids)

dataframes = {}

# Extract records for each available RecordSet
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for RecordSet '@id': {record_set_id}, shape: {dataframes[record_set_id].shape}")
    else:
        print(f"[Warning] RecordSet '@id': {record_set_id} has no records or could not be loaded.")

# Pick the first available DataFrame for further analysis (customize if you know which is main)
selected_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        selected_record_set_id = rid
        break

if selected_record_set_id is not None:
    print(f"\nColumns in DataFrame for RecordSet '@id': {selected_record_set_id}")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    print("No record set contains tabular data in this dataset.")

## 4. Exploratory Data Analysis (EDA)

Basic EDA including filtering numeric fields, normalizing, and grouping.

**Note:**
- Please update `numeric_field_id` and `group_field_id` below to valid field `@id` values from the chosen record set (see printed fields above) for more meaningful analysis.

In [ ]:
# EDA only proceeds if a DataFrame is loaded
if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]

    # --- Configure these:
    # Choose a numeric field '@id' (see field list printed in overview section)
    numeric_field_id = None
    group_field_id = None

    # Fields from metadata may help select the field ids
    for rs in metadata.record_sets:
        if rs.id == selected_record_set_id:
            for field in rs.fields:
                if hasattr(field, 'data_type') and field.data_type in ['Number', 'Float', 'Integer']:
                    numeric_field_id = field.id  # pick first detected numeric field
                if not group_field_id and ('accession' in field.name.lower() or 'name' in field.name.lower()):
                    group_field_id = field.id
            break

    print(f"Numeric field chosen for EDA: {numeric_field_id}")
    print(f"Group field chosen for EDA: {group_field_id}")

    # Fallback: pick first or raise exception
    if numeric_field_id is None:
        numeric_field_id = df.select_dtypes(include=['number']).columns[0] if len(df.select_dtypes(include=['number']).columns) else None
    if group_field_id is None and len(df.columns) > 0:
        group_field_id = df.columns[0]

    print(f"(Actual) Numeric field used: {numeric_field_id}")
    print(f"(Actual) Group field used: {group_field_id}")

    # Proceed with numeric analysis if we have at least one numeric field
    if numeric_field_id is not None and numeric_field_id in df.columns:
        # Remove missing and non-numeric entries
        df_eda = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()].copy()
        df_eda[numeric_field_id] = pd.to_numeric(df_eda[numeric_field_id], errors='coerce')

        # Example filter threshold: use the 75th percentile
        threshold = df_eda[numeric_field_id].quantile(0.75)
        filtered_df = df_eda[df_eda[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 25%):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )

        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a group field (e.g., protein accession/name) if available
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean):")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print("No record set available for EDA.")

## 5. Visualization

Visualize data distributions or basic relationships for the chosen numeric and categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df_eda[numeric_field_id], kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field_id}' in RecordSet '@id': {selected_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    # Boxplot by group field if categorical
    if group_field_id is not None and group_field_id in df_eda.columns and df_eda[group_field_id].nunique() < 20:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_eda)
        plt.title(f"Boxplot of '{numeric_field_id}' grouped by '{group_field_id}'")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization could not be generated due to missing numeric/group fields.")

## 6. Conclusion

In this notebook, we:
- Loaded the FAIR<sup>2</sup> dataset Croissant schema and metadata with `mlcroissant`.
- Examined available record sets and fields using their `@id` references.
- Loaded data into Pandas DataFrames and performed basic exploratory data analysis and visualization leveraging dynamically selected fields.

Explore further by consulting the Croissant schema or field descriptions and adjusting the field IDs for custom numeric or categorical analyses.